# Figure1b volcano


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm
from matplotlib.lines import Line2D
from adjustText import adjust_text
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")

INPUT_CSV = "results/permutation_full/sle_vs_hc/slehc_gene_level_clean_summary.csv"
OUTPUT_DIR = "results/permutation_volcano_publication/sle_vs_hc"

STAT_NAME = "fisher"
PVAL_COL = "fisher_perm_p"
QVAL_COL = "fisher_qval"
EFFECT_COL = "z_score"
PROP_COL = "prop_high_case_donors"

Q_THRESHOLD = 0.1
PROP_CAP = 0.10
CMAP_RED = "Reds"

FIGURE_DPI = 300
LABEL_FONTSIZE = 8
AXIS_FONTSIZE = 11
TITLE_FONTSIZE = 12

CORE_GENES = [
    "YTHDF2", "GPATCH4", "ZBTB38", "FAM222B", "SHROOM4",
    "KCNRG", "SNRNP70", "DIDO1", "RSPH1",
]
EXTENDED_GENES = CORE_GENES + [
    "CCDC137", "C16orf96", "H3F3C", "SIPA1L3", "FIP1L1",
    "CHD9", "ABI3BP", "P4HA2",
]

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
    "font.size": 10,
    "axes.linewidth": 1.0,
    "axes.labelsize": AXIS_FONTSIZE,
    "axes.titlesize": TITLE_FONTSIZE,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "figure.dpi": 100,
    "savefig.dpi": FIGURE_DPI,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.1,
})

In [ ]:
def plot_volcano(df, annotate_genes, save_tag, figsize=(8, 6.5)):

    plot_df = df.dropna(subset=[EFFECT_COL, PVAL_COL]).copy()

    pvals = np.clip(plot_df[PVAL_COL].values, 1e-10, 1.0)
    plot_df["neg_log10_p"] = -np.log10(pvals)

    if QVAL_COL in plot_df.columns and plot_df[QVAL_COL].notna().any():
        plot_df["is_sig"] = plot_df[QVAL_COL] <= Q_THRESHOLD
    else:
        _, qvals, _, _ = multipletests(pvals, method="fdr_bh")
        plot_df[QVAL_COL] = qvals
        plot_df["is_sig"] = qvals <= Q_THRESHOLD

    has_prop = PROP_COL in plot_df.columns and plot_df[PROP_COL].notna().any()
    cmap_r = cm.get_cmap(CMAP_RED)

    effect_vals = plot_df[EFFECT_COL].values
    is_sig = plot_df["is_sig"].values
    colors = []
    point_types = []

    for i, (z, row_idx) in enumerate(zip(effect_vals, plot_df.index)):
        if not is_sig[i]:
            colors.append("lightgray")
            point_types.append("ns")
        elif z > 0:
            point_types.append("sig_pos")
            if has_prop:
                prop = plot_df.loc[row_idx, PROP_COL]
                if pd.isna(prop):
                    prop = 0.0
                prop_scaled = np.clip(prop, 0.0, PROP_CAP) / PROP_CAP
                colors.append(cmap_r(0.15 + 0.85 * prop_scaled))
            else:
                colors.append("firebrick")
        else:
            colors.append("steelblue")
            point_types.append("sig_neg")

    point_types = np.array(point_types)

    fig, ax = plt.subplots(figsize=figsize)

    ns = point_types == "ns"
    if ns.sum():
        ax.scatter(effect_vals[ns], plot_df["neg_log10_p"].values[ns],
                   c="lightgray", s=10, alpha=0.4, edgecolors="none",
                   zorder=1, rasterized=True)

    neg = point_types == "sig_neg"
    if neg.sum():
        ax.scatter(effect_vals[neg], plot_df["neg_log10_p"].values[neg],
                   c="steelblue", s=25, alpha=0.8,
                   edgecolors="white", linewidths=0.5, zorder=2)

    pos = point_types == "sig_pos"
    if pos.sum():
        pos_colors = [colors[i] for i in range(len(colors)) if pos[i]]
        ax.scatter(effect_vals[pos], plot_df["neg_log10_p"].values[pos],
                   c=pos_colors, s=25, alpha=0.85,
                   edgecolors="white", linewidths=0.5, zorder=3)

    ax.axvline(x=0, color="black", linestyle="-", linewidth=0.8, alpha=0.6)
    for p_thresh, ls, lw in [(0.05, ":", 0.6), (0.01, "--", 0.6), (0.001, "-", 0.6)]:
        ax.axhline(y=-np.log10(p_thresh), color="gray", linestyle=ls,
                   linewidth=lw, alpha=0.4)

    ax.set_xlabel("Z-score", fontsize=AXIS_FONTSIZE, fontweight="medium")
    ax.set_ylabel(r"$-\log_{10}$(permutation p-value)", fontsize=AXIS_FONTSIZE,
                  fontweight="medium")
    ax.set_title("Fisher Combined\n(Red intensity = % SLE donors with z ≥ 4.5)",
                 fontsize=TITLE_FONTSIZE, fontweight="bold", pad=10)

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(1.0)
    ax.spines["bottom"].set_linewidth(1.0)
    ax.grid(True, linestyle="--", alpha=0.15, zorder=0)

    if has_prop and pos.sum():
        sm = cm.ScalarMappable(cmap=cmap_r, norm=Normalize(vmin=0, vmax=PROP_CAP))
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, location="right", shrink=0.5,
                            pad=0.02, aspect=20)
        cbar.set_label("Prop. SLE donors with z ≥ 4.5", fontsize=9)
        ticks = np.linspace(0, PROP_CAP, 5)
        tick_labels = [f"{v*100:.0f}%" for v in ticks]
        tick_labels[-1] = f"≥{PROP_CAP*100:.0f}%"
        cbar.set_ticks(ticks)
        cbar.set_ticklabels(tick_labels)

    sig_label = f"FDR < {Q_THRESHOLD}"
    legend_elements = [
        Line2D([0], [0], marker="o", color="w", markerfacecolor=cmap_r(0.8),
               markersize=8, markeredgecolor="white", markeredgewidth=0.5,
               label=f"SLE-enriched ({sig_label})"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="steelblue",
               markersize=8, markeredgecolor="white", markeredgewidth=0.5,
               label=f"HC-enriched ({sig_label})"),
        Line2D([0], [0], marker="o", color="w", markerfacecolor="lightgray",
               markersize=7, label="Not significant"),
    ]
    ax.legend(handles=legend_elements, loc="lower left",
              frameon=True, framealpha=0.9, edgecolor="gray", fontsize=8)

    n_total = len(plot_df)
    n_sig = is_sig.sum()
    n_pos = pos.sum()
    n_neg = neg.sum()
    summary = (f"n = {n_total:,} genes\n"
               f"Significant ({sig_label}): {n_sig}\n"
               f"  SLE-enriched: {n_pos}\n"
               f"  HC-enriched: {n_neg}")
    ax.text(0.98, 0.02, summary, transform=ax.transAxes,
            ha="right", va="bottom", fontsize=8, family="monospace",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="wheat",
                      edgecolor="gray", alpha=0.8))

    texts = []
    annotated = set()

    for gene in annotate_genes:
        mask = plot_df["gene"] == gene
        if mask.sum() == 0:
            print(f"  [NOTE] Gene '{gene}' not found in data — skipping annotation")
            continue
        row = plot_df.loc[mask].iloc[0]
        row_idx = plot_df.index[mask][0]
        pt = point_types[plot_df.index.get_loc(row_idx)]

        if pt == "sig_pos":
            color = cmap_r(0.9)
            fw = "bold"
        elif pt == "sig_neg":
            color = "steelblue"
            fw = "bold"
        else:
            color = "dimgray"
            fw = "normal"

        txt = ax.text(row[EFFECT_COL], row["neg_log10_p"], row["gene"],
                      fontsize=LABEL_FONTSIZE, ha="left", va="bottom",
                      fontweight=fw, color=color, zorder=10)
        texts.append(txt)
        annotated.add(gene)

    if texts:
        try:
            adjust_text(
                texts, ax=ax,
                expand_points=(1.5, 1.8),
                expand_text=(1.2, 1.4),
                force_points=(0.5, 0.8),
                force_text=(0.4, 0.6),
                arrowprops=dict(arrowstyle="-", color="gray", lw=0.5, alpha=0.6),
                only_move={"points": "y", "texts": "xy"},
                lim=500,
            )
        except Exception as e:
            print(f"  [WARN] adjust_text: {e}")

    plt.tight_layout()

    import os
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    png_path = os.path.join(OUTPUT_DIR, f"volcano_fisher_zscore_{save_tag}.png")
    pdf_path = png_path.replace(".png", ".pdf")
    fig.savefig(png_path, dpi=FIGURE_DPI, bbox_inches="tight", facecolor="white")
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight", facecolor="white")
    print(f"  Saved: {png_path}")
    print(f"  Saved: {pdf_path}")
    plt.close()
    return fig

In [ ]:
print("Volcano Plot — Fisher Combined, Z-score axis")

df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df):,} genes from {INPUT_CSV}")
print(f"Columns: {list(df.columns)}\n")

if QVAL_COL not in df.columns or df[QVAL_COL].isna().all():
    print("Computing BH FDR for fisher_perm_p...")
    pvals = df[PVAL_COL].values
    mask = np.isfinite(pvals)
    df[QVAL_COL] = np.nan
    if mask.sum():
        _, qvals, _, _ = multipletests(pvals[mask], method="fdr_bh")
        df.loc[df.index[mask], QVAL_COL] = qvals

n_sig = (df[QVAL_COL] <= Q_THRESHOLD).sum()
print(f"Significant genes (FDR < {Q_THRESHOLD}): {n_sig}\n")

print("--- Version 1: core annotations ---")
plot_volcano(df, CORE_GENES, save_tag="core")

print("\n--- Version 2: extended annotations ---")
plot_volcano(df, EXTENDED_GENES, save_tag="extended")